# 조건부 그래프 — 수강생 실습본

원본: `src/raw/Chapter01/04-langgraph/02_conditional_edge.py`  
대응 슬라이드: `docs/raw/Chapter01/day02/2-1-langgraph-theory-and-advanced-features.pdf`

**학습 목표:** 언어 감지 → 라우팅 → 한국어/영어 노드의 각 단계를 원본 코드에서 찾아 설명한다.

**실행 전:** 언어 감지와 응답 노드에서 LLM을 호출한다. 입력별 분기와 모델 설정을 확인한다.



## 흐름을 먼저 읽기

언어 감지 → 라우팅 → 한국어/영어 노드

핵심은 State와 다음 노드의 결정 방식이다. 입력이 어디서 들어오고 결과가 어디로 이동하는지 화살표를 따라가자. 원본 주석과 TODO를 그대로 둔다. TODO는 CodeMate에게 작업을 지시하고 결과를 검토할 지점으로 읽는다.


## CodeMate 실습 지시

> 다음 코드 흐름을 순서대로 설명해 주세요: 언어 감지 → 라우팅 → 한국어/영어 노드. 라우팅 함수 반환값은 실제 노드 이름과 맞는가? 원본 코드를 바꾸기 전에 근거가 되는 위치와 실행 위험을 알려 주세요.

CodeMate의 제안을 원본의 입력·출력·오류 흐름과 비교하고, 다른 결과가 나오면 다시 지시한다.


## 원본 코드 — 수동 실행

아래 셀은 `.py` 전체를 그대로 보존한다. 위에서 아래로 한 번에 실행되므로 환경과 부작용을 확인한다. **자동 실행/Run All을 사용하지 않는다.**

**이 코드셀의 실행 조건·부작용:** 언어 감지와 응답 노드에서 LLM을 호출한다. 입력별 분기와 모델 설정을 확인한다.



In [ ]:
from typing import Annotated, Literal

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from typing_extensions import TypedDict

from config.llm import get_llm

load_dotenv()

llm = get_llm()


class State(TypedDict):
    messages: Annotated[list, add_messages]
    language: str   # 감지된 언어 ('ko' | 'en')


# ── 노드 정의 ────────────────────────────────────────────────────────────────

def detect_language(state: State) -> dict:
    """마지막 메시지의 언어를 감지해 State에 기록한다."""
    # AIMessage, HumanMessage 
    # 단순히 글자만 띡 있는게 아니라.. 다양한 정보들
    last = state['messages'][-1].content
    # TODO: LLM으로 언어를 감지하고 'ko' 또는 'en' 값을 정해 주세요.
    result = llm.invoke(f'다음 텍스트의 언어를 "ko" 또는 "en" 중 하나로만 판별해: {last}')
    lang = 'ko' if 'ko' in result.content.lower() else 'en'
    return {'language': lang}


def korean_response(state: State) -> dict:
    response = llm.invoke([
        ('system', '반드시 한국어로만 답변하세요.'),
        *state['messages'],
    ])
    return {'messages': [response]}


def english_response(state: State) -> dict:
    response = llm.invoke([
        ('system', 'Respond only in English.'),
        *state['messages'],
    ])
    return {'messages': [response]}


# ── 라우팅 함수 ──────────────────────────────────────────────────────────────
# add_conditional_edges의 두 번째 인자로 전달되는 함수.
# State를 받아 다음에 실행할 노드 이름(또는 END)을 반환한다.
def route_by_language(state: State) -> Literal['korean_response', 'english_response']:
    # 반한값은 조건 분기하고 싶은 노드의 이름
    return 'korean_response' if state.get('language') == 'ko' else 'english_response'


# ── 그래프 조립 ──────────────────────────────────────────────────────────────
builder = StateGraph(State)
builder.add_node('detect_language', detect_language)
builder.add_node('korean_response', korean_response)
builder.add_node('english_response', english_response)

builder.add_edge(START, 'detect_language')

# 조건부 엣지: detect_language 노드 이후 route_by_language 결과에 따라 분기
# TODO: detect_language 이후 조건부 엣지를 추가해 주세요.
    # from -> to
        # 이번에는 언제 어디로 가야할지 도착 노드가 정해지지 않았음.
        # A면 a노드로, B면 b노드로 이동 할 것이다.
            # 그럼 to를 뭘로 결정하느냐? 함수 실행 결과로 결정
builder.add_conditional_edges('detect_language', route_by_language)

# 병렬처리
# builder.add_edge('detect_language', 'korean_response')
# builder.add_edge('detect_language', 'english_response')
builder.add_edge('korean_response', END)
builder.add_edge('english_response', END)

app = builder.compile()


# ── 실행 ─────────────────────────────────────────────────────────────────────
for question in [
    '안녕하세요! LangGraph에 대해 알려주세요.',
    'Hello! Tell me about LangGraph.',
]:
    result = app.invoke({'messages': [HumanMessage(content=question)], 'language': ''})
    print(f'Q: {question}')
    print(f'A: {result["messages"][-1].content}')
    print(f'감지 언어: {result["language"]}')
    print()


## 결과를 검토하기

- 예상 동작: 입력 언어에 따라 한국어 또는 영어 응답 노드로 분기한다.
- 확인 질문: 라우팅 함수 반환값은 실제 노드 이름과 맞는가?
- 오류가 나면 먼저 원본 코드의 어느 단계인지 찾고, CodeMate가 제안한 수정의 이유와 부작용을 설명하게 한다.
